# Solutions — Wanderbricks Bookings & Users (Medium 10)

**Datasets:** `wanderbricks.bookings`, `wanderbricks.users`

In [ ]:
from pyspark.sql import functions as F, types as T
from pyspark.sql import Window

bookings = spark.table("samples.wanderbricks.bookings")
users    = spark.table("samples.wanderbricks.users")

## Solution 1 — Booking Count & Revenue per Country

In [ ]:
result_1 = (
    bookings
    .join(users, on="user_id")
    .groupBy("country")
    .agg(
        F.count("*").alias("booking_count"),
        F.round(F.sum("total_amount"), 2).alias("total_revenue")
    )
    .orderBy(F.col("total_revenue").desc())
)

## Solution 2 — Top 10 Most Active Users by Booking Count

In [ ]:
result_2 = (
    bookings
    .join(users, on="user_id")
    .groupBy("user_id","name","country")
    .agg(F.count("*").alias("booking_count"))
    .orderBy(F.col("booking_count").desc())
    .limit(10)
)

## Solution 3 — Users Who Never Made a Booking (Anti-Join)

In [ ]:
result_3 = (
    users
    .join(bookings.select("user_id").distinct(), on="user_id", how="left_anti")
    .select("user_id","name","country","user_type")
)

## Solution 4 — Business vs Personal Users Comparison

In [ ]:
result_4 = (
    bookings
    .join(users, on="user_id")
    .groupBy("is_business")
    .agg(
        F.round(F.avg("guests_count"), 2).alias("avg_guests"),
        F.round(F.avg(F.datediff("check_out","check_in")), 2).alias("avg_nights"),
        F.round(F.avg("total_amount"), 2).alias("avg_amount"),
        F.count("*").alias("booking_count")
    )
)

## Solution 5 — Users with at Least One Cancelled Booking

In [ ]:
result_5 = (
    bookings
    .filter(F.col("status") == "cancelled")
    .join(users, on="user_id")
    .groupBy("user_id","name","country")
    .agg(F.count("*").alias("cancelled_bookings"))
    .orderBy(F.col("cancelled_bookings").desc())
)

## Solution 6 — Rank Users by Spend within Country (Top 3)

In [ ]:
w = Window.partitionBy("country").orderBy(F.col("total_spend").desc())
result_6 = (
    bookings
    .join(users, on="user_id")
    .groupBy("country","user_id","name")
    .agg(F.round(F.sum("total_amount"), 2).alias("total_spend"))
    .withColumn("rank", F.rank().over(w))
    .filter(F.col("rank") <= 3)
    .orderBy("country","rank")
)

## Solution 7 — Avg Booking Lead Time per User Type

In [ ]:
# Why: lead time = days between created_at (when booking was made) and check_in date
# Note: both bookings and users have created_at — must qualify to avoid ambiguity
result_7 = (
    bookings
    .join(users, on="user_id")
    .withColumn("lead_time_days", F.datediff(F.to_date("check_in"), F.to_date(bookings["created_at"])))
    .filter(F.col("lead_time_days") >= 0)
    .groupBy("user_type")
    .agg(
        F.round(F.avg("lead_time_days"), 1).alias("avg_lead_time_days"),
        F.count("*").alias("booking_count")
    )
    .orderBy("user_type")
)